# Sber sbert_large_mt_nlu_ru — FULL embeddings matrix

GPU ON. Encodes the full 38 360-word ru vocab and saves the matrix as output: `sbert_embeddings_fp16.npy` (38360 x 1024, float16, normalized, rows in vocab.txt order). This is the file the Lambda ranker would use.

In [ ]:
!pip install -q sentence-transformers
import time, numpy as np, urllib.request
from sentence_transformers import SentenceTransformer

In [ ]:
url = 'https://raw.githubusercontent.com/alex-mtl/mao-dao/main/stream-engine/word-data/built/ru/vocab.txt'
vocab = [w.strip() for w in urllib.request.urlopen(url).read().decode('utf-8').splitlines() if w.strip()]
print('vocab', len(vocab))

In [ ]:
model = SentenceTransformer('ai-forever/sbert_large_mt_nlu_ru')
print('dim', model.get_embedding_dimension())
t0 = time.time()
emb = model.encode(vocab, batch_size=256, normalize_embeddings=True, show_progress_bar=True)
print(f'vocab encoded in {time.time()-t0:.0f}s shape {emb.shape}')
emb16 = np.asarray(emb, dtype=np.float16)
np.save('sbert_embeddings_fp16.npy', emb16)
import os
print('saved sbert_embeddings_fp16.npy', os.path.getsize('sbert_embeddings_fp16.npy'), 'bytes')
# sanity: reload and check first rows
chk = np.load('sbert_embeddings_fp16.npy')
print('reloaded', chk.shape, chk.dtype, 'norm row0', float(np.linalg.norm(chk[0].astype(np.float32))))